In [1]:
!pip install pytorch_lightning torchmetrics

In [2]:
import random
import re
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pytorch_lightning as pl

from torch.utils.data import Dataset, DataLoader
from torchmetrics import AUROC
from sklearn.model_selection import train_test_split
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

In [3]:
train_csv: str = "train.csv"
test_csv: str = "test.csv"
submission_csv: str = "submission.csv"

model_name: str = "cointegrated/rubert-tiny2"

seed: int = 42
epochs: int = 5
batch_size: int = 64
max_len: int = 128
freeze_epochs: int = 1

lr: float = 2e-5
weight_decay: float = 0.01
num_workers: int = 2


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(42)

In [4]:
def pick_accelerator() -> str:
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"

In [5]:
# заменяем ссылки/юзеров/числа на спец токены, чтобы модель не пыталась их запоминать
TOKEN_URL = "[URL]"
TOKEN_USER = "[USER]"
TOKEN_NUM = "[NUM]"


def normalize_for_hf(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = re.sub(r"http\S+|www\.\S+", f" {TOKEN_URL} ", text)
    text = re.sub(r"@\w+", f" {TOKEN_USER} ", text)
    text = re.sub(r"\d+", f" {TOKEN_NUM} ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [6]:
# Датасет хранит только тексты и метки
# Токенизацию делаем в collate-функции батча, чтобы включить dynamic padding
class HFDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = list(texts)
        self.labels = None if labels is None else np.asarray(labels, dtype=np.float32)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = normalize_for_hf(self.texts[idx])
        if self.labels is None:
            return {"text": text}
        return {"text": text, "labels": torch.tensor(self.labels[idx]).float()}


def make_collate_fn(tokenizer, max_len: int, with_labels: bool):
    def collate(batch):
        texts = [x["text"] for x in batch]
        enc = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=max_len,
            return_tensors="pt",
        )
        out = {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"]}
        if with_labels:
            out["labels"] = torch.stack([x["labels"] for x in batch]).view(-1, 1)
        return out

    return collate

In [7]:
# Берем предобученный энкодер (RuBERT tiny) + простую линейную голову
# Морозим энкодер на 1ой эпохе , чтобы не сломать резкими градиентами
class HFTransformerClassifier(pl.LightningModule):
    def __init__(
        self,
        model_name: str,
        lr: float,
        weight_decay: float,
        pos_weight: float,
        freeze_epochs: int,
    ):
        super().__init__()
        self.save_hyperparameters()

        from transformers import AutoModel

        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = getattr(self.encoder.config, "hidden_size", None)
        if hidden is None:
            raise ValueError("Could not infer hidden_size from encoder config")

        self.dropout = nn.Dropout(getattr(self.encoder.config, "hidden_dropout_prob", 0.1))
        self.head = nn.Linear(hidden, 1)

        self.register_buffer("pos_weight", torch.tensor([pos_weight], dtype=torch.float))
        self.criterion = nn.BCEWithLogitsLoss(pos_weight=self.pos_weight)
        self.val_auroc = AUROC(task="binary")

    def on_train_epoch_start(self):
        should_freeze = self.current_epoch < int(self.hparams.freeze_epochs)
        for p in self.encoder.parameters():
            p.requires_grad = not should_freeze

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = getattr(out, "pooler_output", None)
        if pooled is None:
            pooled = out.last_hidden_state[:, 0]
        pooled = self.dropout(pooled)
        return self.head(pooled)

    def _step(self, batch, stage: str):
        logits = self(batch["input_ids"], batch["attention_mask"])
        loss = self.criterion(logits, batch["labels"])

        if stage == "val":
            probs = torch.sigmoid(logits).view(-1)
            y_long = batch["labels"].view(-1).long()
            self.val_auroc(probs, y_long)
            self.log("val_auc", self.val_auroc, prog_bar=True)

        self.log(f"{stage}_loss", loss, prog_bar=True)
        return loss

    def training_step(self, batch, batch_idx):
        return self._step(batch, "train")

    def validation_step(self, batch, batch_idx):
        return self._step(batch, "val")

    def configure_optimizers(self):
        no_decay = ["bias", "LayerNorm.weight", "layer_norm.weight"]
        grouped = []
        for n, p in self.named_parameters():
            if not p.requires_grad:
                continue
            wd = 0.0 if any(nd in n for nd in no_decay) else float(self.hparams.weight_decay)
            grouped.append({"params": [p], "weight_decay": wd})

        opt = torch.optim.AdamW(grouped, lr=float(self.hparams.lr))
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            opt, mode="max", factor=0.5, patience=1
        )
        return {"optimizer": opt, "lr_scheduler": {"scheduler": scheduler, "monitor": "val_auc"}}

In [8]:
@torch.no_grad()
def predict_proba(df_test: pd.DataFrame, model: pl.LightningModule, tokenizer, device: str) -> np.ndarray:
    ds = HFDataset(df_test["text"].values, labels=None)
    dl = DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        collate_fn=make_collate_fn(tokenizer, max_len=max_len, with_labels=False),
    )
    model.eval()
    model.to(device)

    probs_all = []
    for batch in dl:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        probs_all.append(torch.sigmoid(logits).squeeze(1).detach().cpu().numpy())
    return np.concatenate(probs_all, axis=0)

In [9]:
accelerator = pick_accelerator()
print(f"Using accelerator: {accelerator}")

df = pd.read_csv(train_csv)
df_test = pd.read_csv(test_csv)

df_train, df_temp = train_test_split(
    df, test_size=0.2, random_state=seed, stratify=df["score"]
)
df_val, df_holdout = train_test_split(
    df_temp, test_size=0.25, random_state=seed, stratify=df_temp["score"]
)

Using accelerator: cuda


In [10]:
# считаем pos_weight для BCEWithLogitsLoss, потому что классы несбалансированы
pos = float(df_train["score"].sum())
neg = float(len(df_train) - pos)
pos_weight = neg / max(pos, 1.0)
print(f"pos_weight: {pos_weight:.4f}")

pos_weight: 4.5664


In [11]:
# Загружаем HF tokenizer
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

train_ds = HFDataset(df_train["text"].values, df_train["score"].values)
val_ds = HFDataset(df_val["text"].values, df_val["score"].values)

train_dl = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    collate_fn=make_collate_fn(tokenizer, max_len=max_len, with_labels=True),
)
val_dl = DataLoader(
    val_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    collate_fn=make_collate_fn(tokenizer, max_len=max_len, with_labels=True),
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [12]:
# создаем LightningModule (энкодер + голова) и передаем параметры обучения
model = HFTransformerClassifier(
    model_name=model_name,
    lr=lr,
    weight_decay=weight_decay,
    pos_weight=pos_weight,
    freeze_epochs=freeze_epochs,
)

callbacks = [
    EarlyStopping(monitor="val_auc", patience=1, mode="max", verbose=True),
    ModelCheckpoint(monitor="val_auc", mode="max", save_top_k=1),
]
trainer = pl.Trainer(
    max_epochs=epochs,
    accelerator=accelerator,
    devices=1,
    gradient_clip_val=1.0,
    log_every_n_steps=20,
    callbacks=callbacks,
    enable_progress_bar=True,
)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores


In [13]:
trainer.fit(model, train_dataloaders=train_dl, val_dataloaders=val_dl)
metrics = trainer.validate(model, dataloaders=val_dl, verbose=False)
print("Validation metrics:", metrics[0])

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ encoder   │ BertModel         │ 29.2 M │ eval  │     0 │
│ 1 │ dropout   │ Dropout           │      0 │ train │     0 │
│ 2 │ head      │ Linear            │    313 │ train │     0 │
│ 3 │ criterion │ BCEWithLogitsLoss │      0 │ train │     0 │
│ 4 │ val_auroc │ BinaryAUROC       │      0 │ train │     0 │
└───┴───────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 29.2 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 29.2 M                                                                                               
Total estimated model params size (MB): 116                                                                        
Modules in train mode: 4                                                                                           
Modules in eval mode: 66                                                                                           
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/loops/fit_loop.py:534: Found 66 module(s) in eval mode at
the start of training. This may lead to unexpected behavior during training. If this is intentional, you can ignore
this warning.

INFO:pytorch_lightning.callbacks.early_stopping:Metric val_auc improved. New best score: 0.781
INFO:pytorch_lightning.callbacks.early_stopping:Metric val_auc improved by 0.211 >= min_delta = 0.0. New best score: 0.992
INFO:pytorch_lightning.callbacks.early_stopping:Metric val_auc improved by 0.001 >= min_delta = 0.0. New best score: 0.994
INFO:pytorch_lightning.callbacks.early_stopping:Monitored metric val_auc did not improve in the last 1 records. Best score: 0.994. Signaling Trainer to stop.


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

Validation metrics: {'val_auc': 0.9935873746871948, 'val_loss': 0.2261289656162262}


In [14]:
# сохраняем в submission.csv
probs = predict_proba(df_test, model, tokenizer, device=accelerator)
submission = pd.DataFrame({"id": df_test["id"].values, "score": probs.astype(float)})
submission.to_csv(submission_csv, index=False)
print("Saved:", submission_csv)
print(submission.head())

Saved: submission.csv
       id     score
0  152591  0.000570
1  108771  0.999116
2  198853  0.000569
3  194736  0.000864
4   88110  0.003369
